# Bollinger Breakout Strategy Demonstration

This walkthrough mirrors `qc_projects/main_bollinger.py`, originally built for QuantConnect Lean. We will:
- Use Yahoo Finance daily candles for SPY between 2020-01-01 and 2022-01-01
- Recreate the 20-period Bollinger Bands, 14-period ATR, and 20-period volume SMA stack
- Apply the same breakout/volume confirmation entry plus band-based exit rules
- Review signals, trade stats, and visual diagnostics

## Step 1: Imports and QC-Aligned Parameters

We keep every knob identical to the Lean script: 20-period Bollinger Bands (2σ), 14-period ATR for sizing, 20-period volume SMA for confirmation, and a 25-bar warmup before signals fire.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from pathlib import Path

pd.set_option("display.max_columns", None)

symbol = "SPY"
start_date = "2020-01-01"
end_date = "2022-01-01"
bb_period = 20
bb_std = 2.0
atr_period = 14
volume_window = 20
volume_threshold = 1.2  # Lean script multiplies avg volume by 1.2
warmup_bars = 25

print(f"Symbol: {symbol} | Window: {start_date} → {end_date}")
print(f"Bollinger: {bb_period} periods @ {bb_std}σ | ATR period: {atr_period}")
print(f"Volume SMA: {volume_window} periods | Volume threshold: {volume_threshold}×")

Symbol: SPY | Window: 2020-01-01 → 2022-01-01
Bollinger: 20 periods @ 2.0σ | ATR period: 14
Volume SMA: 20 periods | Volume threshold: 1.2×


## Step 2: Pull Daily OHLCV for SPY

QuantConnect Lean sources data from its own feeds; here we substitute Yahoo Finance daily candles for the same window and keep the canonical OHLCV columns.

In [5]:
price_df = yf.download(
    symbol,
    start=start_date,
    end=end_date,
    interval="1d",
    progress=False,
    multi_level_index=False,
)
price_df = price_df.dropna().tz_localize(None)
price_df.index.name = "date"

print(f"Fetched {len(price_df):,} daily bars")
price_df.head()

Fetched 505 daily bars


C:\Users\User\AppData\Local\Temp\ipykernel_49560\471923477.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_df = yf.download(


,Close,High,Low,Open,Volume
date,,,,,
2020-01-02,297.699005,297.717350,295.554718,296.480254,59151200
2020-01-03,295.444733,296.571870,294.244299,294.299278,77709700
2020-01-06,296.571869,296.654338,293.566170,293.685302,55653900
2020-01-07,295.738037,296.480289,295.289000,296.003762,40496400
2020-01-08,297.314178,298.532930,295.683052,295.930460,68296000


## Step 3: Rebuild Bollinger Bands, ATR, and Volume SMA

The Lean strategy feeds three indicators: Bollinger Bands(20,2), ATR(14), and Volume SMA(20). We reproduce the same math with pandas rolling windows and keep intermediary columns for inspection.

In [6]:
features = price_df.copy()
features["mid_band"] = features["Close"].rolling(bb_period).mean()
features["band_std"] = features["Close"].rolling(bb_period).std(ddof=0)
features["upper_band"] = features["mid_band"] + bb_std * features["band_std"]
features["lower_band"] = features["mid_band"] - bb_std * features["band_std"]

features["prev_close"] = features["Close"].shift(1)
features["tr1"] = features["High"] - features["Low"]
features["tr2"] = (features["High"] - features["prev_close"]).abs()
features["tr3"] = (features["Low"] - features["prev_close"]).abs()
features["true_range"] = features[["tr1", "tr2", "tr3"]].max(axis=1)
features["atr"] = features["true_range"].rolling(atr_period).mean()

features["volume_sma"] = features["Volume"].rolling(volume_window).mean()
features["volume_ratio"] = features["Volume"] / features["volume_sma"]
features["volume_confirmed"] = features["volume_ratio"] > volume_threshold

feature_view = features[[
    "Close",
    "upper_band",
    "mid_band",
    "lower_band",
    "atr",
    "Volume",
    "volume_sma",
    "volume_ratio",
    "volume_confirmed",
]].dropna()

feature_view.tail()

,Close,upper_band,mid_band,lower_band,atr,Volume,volume_sma,volume_ratio,volume_confirmed
date,,,,,,,,,
2021-12-27,451.449524,450.773624,437.625192,424.476760,6.107973,56808600,96995915.0,0.585680,False
2021-12-28,451.080566,452.681870,438.282443,423.883016,5.605001,47274600,95046205.0,0.497385,False
2021-12-29,451.657562,454.309236,439.394600,424.479964,5.569939,54503000,90343375.0,0.603287,False
2021-12-30,450.408997,454.702814,440.682809,426.662805,5.559954,55329000,86512865.0,0.639546,False
2021-12-31,449.273926,455.367938,441.589066,427.810193,5.401568,65237400,83392845.0,0.782290,False


## Step 4: Replay the Breakout Logic

Lean's `OnData` handler waits for indicators, checks the breakout (close above upper band), confirms volume, sizes via ATR, and exits when price tags either the middle or lower band. We replicate the same state machine below.

In [7]:
ready_frame = feature_view.copy()
ready_frame = ready_frame.assign(bar_index=range(len(ready_frame)))

trades = []
entry_markers = []
exit_markers = []
position = None

for i, (as_of, row) in enumerate(ready_frame.iterrows()):
    if i < warmup_bars:
        continue  # mimic Lean warmup requirement
    price = row["Close"]
    atr_value = row["atr"]
    upper = row["upper_band"]
    middle = row["mid_band"]
    lower = row["lower_band"]
    volume_ok = bool(row["volume_confirmed"])

    if position is None:
        if not np.isfinite([upper, atr_value, row["volume_ratio"]]).all():
            continue
        if price > upper and volume_ok:
            if atr_value > 0:
                position_size = min(0.95, (2.0 / atr_value) * 0.1)
            else:
                position_size = 0.5
            position = {
                "entry_index": i,
                "entry_date": as_of,
                "entry_price": price,
                "size": position_size,
                "entry_volume_ratio": row["volume_ratio"],
            }
            entry_markers.append({"date": as_of, "price": price})
    else:
        exit_condition = (price <= lower) or (price <= middle)
        if exit_condition:
            ret_pct = (price - position["entry_price"]) / position["entry_price"] * 100
            bars_held = i - position["entry_index"]
            trades.append({
                "entry_date": position["entry_date"].date().isoformat(),
                "exit_date": as_of.date().isoformat(),
                "entry_price": round(position["entry_price"], 2),
                "exit_price": round(price, 2),
                "return_pct": round(ret_pct, 2),
                "bars_held": bars_held,
                "position_size": round(position["size"], 4),
                "volume_ratio": round(position["entry_volume_ratio"], 2),
            })
            exit_markers.append({"date": as_of, "price": price})
            position = None

# Force liquidation at the final bar if still invested
if position is not None:
    final_row = ready_frame.iloc[-1]
    final_price = final_row["Close"]
    ret_pct = (final_price - position["entry_price"]) / position["entry_price"] * 100
    bars_held = len(ready_frame) - position["entry_index"]
    trades.append({
        "entry_date": position["entry_date"].date().isoformat(),
        "exit_date": ready_frame.index[-1].date().isoformat(),
        "entry_price": round(position["entry_price"], 2),
        "exit_price": round(final_price, 2),
        "return_pct": round(ret_pct, 2),
        "bars_held": bars_held,
        "position_size": round(position["size"], 4),
        "volume_ratio": round(position["entry_volume_ratio"], 2),
        "forced_exit": True,
    })
    exit_markers.append({"date": ready_frame.index[-1], "price": final_price})
    position = None

trades_df = pd.DataFrame(trades)
print(f"Completed simulation with {len(trades_df)} trades")
trades_df.head() if not trades_df.empty else "No trades captured"

Completed simulation with 3 trades


,entry_date,exit_date,entry_price,exit_price,return_pct,bars_held,position_size,volume_ratio
0,2020-06-05,2020-06-11,294.35,277.09,-5.87,4,0.0401,1.56
1,2020-09-02,2020-09-04,331.16,317.15,-4.23,2,0.0728,1.35
2,2020-12-31,2021-01-27,349.01,349.50,0.14,17,0.0527,1.28


## Step 5: Summarize Trade Stats

We compute aggregate performance (trade count, win rate, return distribution) and keep the per-trade table for deeper inspection or export to reports.

In [8]:
if trades_df.empty:
    summary = {
        "total_trades": 0,
        "win_rate_pct": 0.0,
        "avg_return_pct": 0.0,
        "median_return_pct": 0.0,
        "avg_bars": 0.0,
    }
else:
    summary = {
        "total_trades": len(trades_df),
        "win_rate_pct": round((trades_df["return_pct"] > 0).mean() * 100, 1),
        "avg_return_pct": round(trades_df["return_pct"].mean(), 2),
        "median_return_pct": round(trades_df["return_pct"].median(), 2),
        "avg_bars": round(trades_df["bars_held"].mean(), 1),
    }

summary

{'total_trades': 3,
 'win_rate_pct': np.float64(33.3),
 'avg_return_pct': np.float64(-3.32),
 'median_return_pct': np.float64(-4.23),
 'avg_bars': np.float64(7.7)}

## Step 6: Visual Diagnostics

Overlay price with Bollinger Bands plus entry/exit markers, and save the interactive figure under `graphs/` for comparison with other strategy notebooks.

In [9]:
graphs_dir = Path("graphs")
graphs_dir.mkdir(exist_ok=True)

plot_frame = ready_frame.copy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=plot_frame.index, y=plot_frame["Close"], name="Close", line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=plot_frame.index, y=plot_frame["upper_band"], name="Upper Band", line=dict(color="#ff7f0e", dash="dash")))
fig.add_trace(go.Scatter(x=plot_frame.index, y=plot_frame["mid_band"], name="Mid Band", line=dict(color="#2ca02c", dash="dot")))
fig.add_trace(go.Scatter(x=plot_frame.index, y=plot_frame["lower_band"], name="Lower Band", line=dict(color="#d62728", dash="dash")))

if entry_markers:
    fig.add_trace(
        go.Scatter(
            x=[m["date"] for m in entry_markers],
            y=[m["price"] for m in entry_markers],
            mode="markers",
            marker=dict(symbol="triangle-up", size=12, color="#2ca02c"),
            name="Entries",
        )
    )

if exit_markers:
    fig.add_trace(
        go.Scatter(
            x=[m["date"] for m in exit_markers],
            y=[m["price"] for m in exit_markers],
            mode="markers",
            marker=dict(symbol="triangle-down", size=12, color="#d62728"),
            name="Exits",
        )
    )

fig.update_layout(title="Bollinger Breakout Strategy (QC Parity)", yaxis_title="Price", height=500)

output_path = graphs_dir / "bollinger_breakout_signals.html"
fig.write_html(output_path)
fig.show()
print(f"Saved interactive chart to {output_path}")

Saved interactive chart to graphs\bollinger_breakout_signals.html


## Step 7: Inspect Trade Log

Lean logs entries/exits via `Debug` statements; here we expose the structured trade table directly for downstream analysis or reporting.

In [10]:
if trades_df.empty:
    print("No trades generated under the current configuration.")
else:
    display(trades_df.tail(10))

,entry_date,exit_date,entry_price,exit_price,return_pct,bars_held,position_size,volume_ratio
0,2020-06-05,2020-06-11,294.35,277.09,-5.87,4,0.0401,1.56
1,2020-09-02,2020-09-04,331.16,317.15,-4.23,2,0.0728,1.35
2,2020-12-31,2021-01-27,349.01,349.50,0.14,17,0.0527,1.28


## Recap

- This notebook mirrors the Lean logic in `qc_projects/main_bollinger.py`, so you can validate tweaks locally before porting back to QuantConnect.
- Adjust the parameters cell (symbol, band/ATR lengths, volume multiplier) to explore other assets or timeframes.
- The resulting `trades_df`, `summary`, and HTML chart in `graphs/` can feed the broader analysis flows under `docs/` or the Streamlit dashboard.